In [ ]:
import os
print(os.getcwd())

In [ ]:
import sys
sys.path.append('/mmpatil/MEA_Analysis_FEB25/IPNAnalysis/StimAnalysis')
from StimAnalysis import StimulationAnalysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spikeinterface
import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.sorters as ss
import spikeinterface.comparison as sc
import spikeinterface.widgets as sw
import spikeinterface.postprocessing as sp
import spikeinterface.preprocessing as spre
import spikeinterface.qualitymetrics as qm
#import helper_functions as helper

In [ ]:
#Reading the file, BP filtering
local_path= '/mnt/disk20tb/PrimaryNeuronData/Maxtwo/Stimulation_MaxOnePlus_MP/250423/P002820/Trace_20250423_16_49_17.raw.h5' #network data from chip 16848

recording1 = se.read_maxwell(local_path)
freq_min = 300
freq_max = 4500


#recording = si.ConcatenateSegmentRecording([recording1,recording2])
channel_ids = recording1.get_channel_ids()
fs = recording1.get_sampling_frequency()
num_chan = recording1.get_num_channels()
num_seg = recording1.get_num_segments()
total_recording = recording1.get_total_duration()


#print('Channel ids:', channel_ids)
print('Sampling frequency:', fs)
print('Number of channels:', num_chan)
print('Number of segments:', num_seg)
print(f"total_recording: {total_recording} s")

recording_bp = spre.bandpass_filter(recording1, freq_min=freq_min, freq_max=freq_max)

recodring_cmr = spre.common_reference(recording_bp, reference='global', operator='median')
#recording_chunk = recodring_cmr.frame_slice(start_frame= 1*fs,end_frame=425*fs)
recording_chunk = recodring_cmr.frame_slice(start_frame= 0*fs,end_frame=900*fs)
print(f"chunk duration: {recording_chunk.get_total_duration()} s")


In [ ]:
# Extract channel IDs and 2D location coordinates
channel_ids =np.array([int(x) for x in recording_chunk.get_channel_ids()])
locs = recording_chunk.get_channel_locations()
channels_to_highlight = [38]# Channels to highlight



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector
import ipywidgets as widgets
from IPython.display import display
import os

# Enable interactive matplotlib
%matplotlib widget
# plt.flush() is not a valid method and has been removed
# Setup figure
fig, ax = plt.subplots()
c = ['gray'] * len(channel_ids)  # Default color for all channels
if channels_to_highlight is not None:
    for ch in channels_to_highlight: 
        if ch in channel_ids:
            idx = np.where(channel_ids == ch)[0][0]
            c[idx] = 'blue'  # Highlight specified channels in red   

sc = ax.scatter(locs[:, 0], locs[:, 1], c=c, label='All Channels', s=10)
ax.set_title("Drag to select a region")
ax.set_xlabel("X (μm)")
ax.set_ylabel("Y (μm)")


# Scatter for selected points
selected_plot = ax.scatter([], [], facecolors='none', edgecolors='red', linewidths=1, s=20, label='Selected Channels')
selected_channels = []

# Create an output widget for displaying messages
output = widgets.Output()

# Selection callback with cumulative selection
def onselect(eclick, erelease):
    global selected_channels
    if eclick.xdata is None or erelease.xdata is None:  # Ignore clicks outside the plot
        return

    x0, y0 = eclick.xdata, eclick.ydata
    x1, y1 = erelease.xdata, erelease.ydata

    mask = (
        (locs[:, 0] > min(x0, x1)) & (locs[:, 0] < max(x0, x1)) &
        (locs[:, 1] > min(y0, y1)) & (locs[:, 1] < max(y0, y1))
    )
    new_selection = channel_ids[mask]

    # Check if Ctrl key is pressed (use event modifiers)
    if eclick.key == 'control':  # Append to existing selection
        selected_channels = np.unique(np.concatenate((selected_channels, new_selection)))
    else:  # Overwrite selection
        selected_channels = new_selection

    selected_plot.set_offsets(locs[np.isin(channel_ids, selected_channels)])
    fig.canvas.draw_idle()

    # Display the selection details in the output widget
    with output:
        output.clear_output()  # Clear previous messages
        print(f"🔴 Selected {len(selected_channels)} channels")
        print(selected_channels)

# Create RectangleSelector
selector = RectangleSelector(
    ax, onselect,
    interactive=True,
    useblit=True,
    button=[1]
)

# Create save button
save_button = widgets.Button(
    description="💾 Save Selected Channels",
    button_style='success',
    tooltip='Save selected channel IDs to file'
)

# Define callback to save to file
def save_selected_channels_to_file(b):
    if len(selected_channels) == 0:
        with output:
            output.clear_output()
            print("No channels selected to save.")
        return

    save_path = "selected_channels.txt"
    with open(save_path, "w") as f:
        f.write(",".join(map(lambda ch: f'"{ch}"', selected_channels)))

    with output:
        output.clear_output()
        print(f"Saved {len(selected_channels)} channel IDs to {os.path.abspath(save_path)}")

# Attach callback and show button
save_button.on_click(save_selected_channels_to_file)

# Display the save button and output widget
display(widgets.VBox([save_button, output]))
plt.xlim([0,4000])
plt.ylim([0,3000])
plt.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt
import importlib
# Reload the spikeinterface.widgets module
importlib.reload(sw)
%matplotlib widget
# Visualize the traces interactively
print("Visualizing traces interactively...")
channel_ids = ["100", "820"]
sw.plot_timeseries(recording_chunk, channel_ids=channel_ids, mode='line', show_channel_ids=True, backend='ipywidgets')

In [ ]:
import spikeinterface.preprocessing as spp
import spikeinterface.extractors as se

# Step 1: Scale the recording (applies gain/offset so output is in µV)
recording_scaled = spp.scale(recording_chunk)

# Step 2: Save the scaled recording as binary (.raw + .json)
se.BinaryRecordingExtractor.write_recording(
    recording=recording_scaled,
    file_paths='/mmpatil/MEA_Analysis_FEB25/IPNAnalysis/StimAnalysis/saved_mmap',
    dtype='float32',  # Saves scaled data in float32
    progress_bar=True,
    n_jobs=32
)

In [ ]:
# Step 3: Load mmap recording
recording_mmap = se.BinaryRecordingExtractor( file_paths='/mmpatil/MEA_Analysis_FEB25/IPNAnalysis/StimAnalysis/saved_mmap..raw',
    sampling_frequency=recording_chunk.get_sampling_frequency(),
    num_channels=recording_chunk.get_num_channels(),
    dtype='float32',  # must match what you saved
    channel_ids=recording_chunk.get_channel_ids() )



In [ ]:

fs = recording_mmap.get_sampling_frequency()
start_frame = int(300 * fs)
end_frame = int(600 * fs)

recording_slice = recording_mmap.frame_slice(start_frame, end_frame)

In [ ]:
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spp
from spikeinterface.sortingcomponents.peak_detection import detect_peaks

from spikeinterface.core.recording_tools import get_noise_levels

import numpy as np

# Construct custom absolute thresholds: 18 µV for each channel
num_channels = recording_chunk.get_num_channels()
manual_abs_thresholds = np.full(num_channels, 1)  # µV

# Now run detect_peaks with custom noise levels
peaks2 = detect_peaks(
    recording=recording_slice,
    method='by_channel',
    peak_sign='neg',
    detect_threshold=10,         # k = 1 → just use manual_abs_thresholds
    noise_levels=manual_abs_thresholds,  # These are already in µV     the data is 
    chunk_duration='300s',
    n_jobs=32,
    progress_bar=True
)

"""
A noise level of 0.00 µV will break the logic of peak detection using k × MAD, because:

Any finite value of detect_threshold × noise becomes zero, and every downward blip becomes a “detected spike”.
"""

In [ ]:
peaks2

In [ ]:
# Get scaled noise levels (in µV)
noise_levels = get_noise_levels(recording_chunk, return_scaled=True)
print(noise_levels.shape)
# Optional: Print estimated noise level for Channel 88
channel_idx = np.where(recording_chunk.get_channel_ids() == "88")[0][0]
print(f"Noise level on Channel 88 = {noise_levels[channel_idx]:.2f} µV")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# --- Parameters ---
channels = ["38","100","820","827","844"]  # List of channel IDs as strings
start_time = 41.34
end_time = 41.54
fs = recording_chunk.get_sampling_frequency()
rec_var = recording_chunk.frame_slice(start_frame=int(300 * fs), end_frame=int(600 * fs))

# --- Extract traces and spike data for each channel ---
plt.close('all')
plt.figure(figsize=(10, len(channels) * 3))  # Adjust figure height dynamically
offset = 100  # Offset for separating traces visually
yticks_positions = []  # Store y-axis positions for labeling
yticks_labels = []  # Store channel names for labeling

for i, channel in enumerate(channels):
    channel_int = int(channel)  # Convert to int for comparison

    # Extract trace
    start_frame = int(start_time * fs)
    end_frame = int(end_time * fs)
    trace = rec_var.get_traces(
        start_frame=start_frame,
        end_frame=end_frame,
        channel_ids=[channel],
        return_scaled=True
    ).flatten()

    # Time axis
    time_axis = np.arange(len(trace)) / fs + start_time

    # Extract spike times for this channel in time window
    channel_idx = np.where(recording_chunk.get_channel_ids() == channel)[0][0]  # Get the integer index of the channel
    channel_peaks = peaks2[peaks2['channel_index'] == channel_idx]
    spike_samples = channel_peaks['sample_index']

    # Keep spikes only within current time window
    mask = (spike_samples >= start_frame) & (spike_samples <= end_frame)
    spike_times_window = spike_samples[mask]
    spike_sample_indices = spike_times_window - start_frame
    spike_amplitudes = trace[spike_sample_indices]

    # Plot trace and spikes
    plt.plot(
        time_axis,
        trace + i * offset,  # Offset for visual separation
        linewidth=0.7,
        label=f"Trace (Channel {channel})"
    )
    if channel =="38" : print(spike_sample_indices)
    # plt.plot(
    #     time_axis[spike_sample_indices],
    #     spike_amplitudes + i * offset,
    #     'rv',
    #     markersize=6,
    #     label=f"Spikes (Channel {channel})"
    # )

    # Add y-axis tick position and label
    yticks_positions.append(i * offset)
    yticks_labels.append(f"Channel {channel}")

# --- Customize plot ---
plt.xlabel("Time (s)")
plt.ylabel("Voltage (µV)")
plt.title("Traces for Multiple Channels with Spikes")
#plt.grid(True)
#plt.legend()
plt.yticks(yticks_positions, yticks_labels)
plt.tight_layout()
plt.savefig("traces_with_spikes.svg", dpi=300,format='svg')
plt.show()

In [ ]:
importlib.reload(plt)
%matplotlib widget
plt.close('all')
fig, ax = plt.subplots(figsize=(8, 8))
si.plot_probe_map(recording_chunk,ax=ax,with_channel_ids=True)
ax.invert_yaxis()
#ax.set_facecolor('white')  # Set the background color to white
ax.axis('off')  # Turn off the axes
fig.show()

In [ ]:
fs = recording_mmap.get_sampling_frequency()
start_frame = int(0 * fs)
end_frame = int(300 * fs)

recording_slice = recording_mmap.frame_slice(start_frame, end_frame)
num_channels = recording_chunk.get_num_channels()
manual_abs_thresholds = np.full(num_channels, 1)  # µV

# Now run detect_peaks with custom noise levels
peaks = detect_peaks(
    recording=recording_slice,
    method='by_channel',
    peak_sign='neg',
    detect_threshold=10,         # k = 1 → just use manual_abs_thresholds
    noise_levels=manual_abs_thresholds,  # These are already in µV     the data is 
    chunk_duration='300s',
    n_jobs=32,
    progress_bar=True
)


In [ ]:
fs = recording_mmap.get_sampling_frequency()
start_frame = int(600 * fs)
end_frame = int(900 * fs)

recording_slice = recording_mmap.frame_slice(start_frame, end_frame)
num_channels = recording_chunk.get_num_channels()
manual_abs_thresholds = np.full(num_channels, 1)  # µV

# Now run detect_peaks with custom noise levels
peaks3 = detect_peaks(
    recording=recording_slice,
    method='by_channel',
    peak_sign='neg',
    detect_threshold=10,         # k = 1 → just use manual_abs_thresholds
    noise_levels=manual_abs_thresholds,  # These are already in µV     the data is 
    chunk_duration='300s',
    n_jobs=32,
    progress_bar=True
)


In [ ]:
import numpy as np

channels, counts = np.unique(peaks2['channel_index'], return_counts=True)
for ch, count in zip(channels, counts):
    print(ch, count)
    print(f"Channel {recording_mmap.get_channel_ids()[ch]} → {count} spikes")

In [ ]:
import numpy as np

channels, counts = np.unique(peaks3['channel_index'], return_counts=True)
for ch, count in zip(channels, counts):
    print(f"Channel {recording_mmap.get_channel_ids()[ch]} → {count} spikes")

In [ ]:
peaks2['sample_index'][peaks2['channel_index']==100]

In [ ]:
import numpy as np

channels, counts = np.unique(peaks3['channel_index'], return_counts=True)
for ch, count in zip(channels, counts):
    print(f"Channel {ch} → {count} spikes")

#statistics of counts

# Basic statistics
print(f"Total active channels: {len(channels)}")
print(f"Total spikes detected: {np.sum(counts)}")
print(f"Mean spike count: {np.mean(counts):.2f}")
print(f"Median spike count: {np.median(counts):.2f}")
print(f"Standard deviation: {np.std(counts):.2f}")
print(f"Min spike count: {np.min(counts)}")
print(f"Max spike count: {np.max(counts)}")

# Optional: percentiles
print(f"25th percentile: {np.percentile(counts, 25):.2f}")
print(f"75th percentile: {np.percentile(counts, 75):.2f}")


In [ ]:
target_channel = "100"
print(recording_mmap.get_channel_ids())
target_channel_idx = np.where(recording_mmap.get_channel_ids()==target_channel)[0][0]
sample_indices = peaks2['sample_index'][peaks2['channel_index'] == target_channel_idx]
sample_indices

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
# Import the detect_peaks function
%matplotlib inline
channels_to_plot = [100] 
fs = recording_chunk.get_sampling_frequency()
start_time = 42.75 # Start time in seconds
end_time = 43.75 # End time in seconds
# Get traces for the specified range
start_frame = int(300*fs)
end_frame =int(600*fs)

recording_slice = recording_chunk.frame_slice(start_frame, end_frame)
traces = recording_slice.get_traces(start_frame=int(start_time*fs), end_frame=int(end_time*fs), segment_index=0,return_scaled=True)
print(traces.shape)
stim_channel ="38"

plt.close('all')
# Adjust figure height dynamically based on the number of channels
plt.figure(figsize=(12, len(channels_to_plot) * 0.5))  # Increase height for better spacing

# Loop through the channels and plot each
yticks_positions = []  # Store y-axis positions for labeling
yticks_labels = []  # Store channel names for labeling
offset = 10
for i, channel in enumerate(channels_to_plot):
    # Get the trace for the current channel
    trace = traces[:, channel]
    print(trace)
    # Detect peaks using the detect_peaks function
    #peaks_sample_inds, _ = detect_peaks_stddev(trace, peak_sign, std_multiplier)
    
    # Convert peaks_sample_inds to integers
    stim_channel_idx = np.where(recording_mmap.get_channel_ids() == stim_channel)[0][0]
    target_channel_idx = np.where(recording_mmap.get_channel_ids() == str(channel))[0][0]
    print(target_channel_idx)
    peaks_sample_inds =peaks2['sample_index'][peaks2['channel_index'] == channel]
    print(peaks_sample_inds)
    stim_sample_inds = peaks2['sample_index'][peaks2['channel_index'] == 38]
    spike_marker_offset = 15
    
    # Plot the trace with increased spacing
    plt.plot(
        trace + i * offset,  # Increase the offset to 200 for better spacing
        label=f'Channel {channel}', 
        rasterized=True,  # Rasterize the line plots for faster rendering
        linewidth=0.5  # Use thinner lines for better performance
    )
    peaks_sample_inds = (peaks_sample_inds[(peaks_sample_inds > int(start_time * fs)) & (peaks_sample_inds < int(end_time * fs))]-int(start_time * fs))
    print(peaks_sample_inds)
    # Mark the detected spikes with red triangles
    plt.plot(
        peaks_sample_inds, 
        trace[peaks_sample_inds] + i * offset + spike_marker_offset,  # Match the increased offset
        'rv',  # Red triangles
        markersize=4, 
        label=f'Spikes Channel {channel}'
    )
    
    # Add y-axis tick position and label
    yticks_positions.append(i * offset)
    yticks_labels.append(f'Channel {channel}')

# Customize the plot
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['bottom'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.xlabel('Time (samples)')
plt.title('Traces for Multiple Channels with Detected Spikes')

# Add y-axis ticks with channel names
plt.yticks(yticks_positions, yticks_labels)

# Save the plot as a PDF
output_path = './savetraces_multiple_channels_with_spikes.pdf'  # Replace with your desired path
plt.show() 
plt.savefig(output_path, format='pdf', bbox_inches='tight', dpi=150)  # Save with high resolution
 # Show the plot
# Close the plot to free memory
plt.close()

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
from helper_functions import detect_peaks_stddev  # Import the detect_peaks function
%matplotlib inline
channels_to_plot = [38,100] 
fs = recording_mmap.get_sampling_frequency()
start_time =300 # Start time in seconds
end_time = 305# End time in seconds
# Get traces for the specified range
traces = recording_mmap.get_traces(start_frame=int(start_time*fs), end_frame=int(end_time*fs), segment_index=0, return_scaled=True)



# Adjust figure height dynamically based on the number of channels
plt.figure(figsize=(12, len(channels_to_plot) * 0.5))  # Increase height for better spacing

# Loop through the channels and plot each
yticks_positions = []  # Store y-axis positions for labeling
yticks_labels = []  # Store channel names for labeling

for i, channel in enumerate(channels_to_plot):
    # Get the trace for the current channel
    trace = traces[:, channel]
    

    
    # Convert peaks_sample_inds to integers
    peaks_sample_inds = peaks['sample_index']
    spike_marker_offset = 100
    
    # Plot the trace with increased spacing
    plt.plot(
        trace + i * 200,  # Increase the offset to 200 for better spacing
        label=f'Channel {channel}', 
        rasterized=True,  # Rasterize the line plots for faster rendering
        linewidth=0.5  # Use thinner lines for better performance
    )
    
    # Mark the detected spikes with red triangles
    plt.plot(
        peaks_sample_inds, 
        trace[peaks_sample_inds] + i * 200 + spike_marker_offset,  # Match the increased offset
        'rv',  # Red triangles
        markersize=4, 
        label=f'Spikes Channel {channel}'
    )
    
    # Add y-axis tick position and label
    yticks_positions.append(i * 200)
    yticks_labels.append(f'Channel {channel}')

# Customize the plot
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['bottom'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.xlabel('Time (samples)')
plt.title('Traces for Multiple Channels with Detected Spikes')

# Add y-axis ticks with channel names
plt.yticks(yticks_positions, yticks_labels)

# Save the plot as a PDF
output_path = './savetraces_multiple_channels_with_spikes.pdf'  # Replace with your desired path
plt.show() 
plt.savefig(output_path, format='pdf', bbox_inches='tight', dpi=150)  # Save with high resolution
 # Show the plot
# Close the plot to free memory
plt.close()


In [ ]:
print(peaks.dtype)

In [ ]:
len(recording_chunk.get_channel_ids())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

def custom_diverging_colormap(n_colors=256):
    # [position, R, G, B]
    control_points = np.array([
        [0.00, 0.0, 0.0, 0.0],    # Black
        [0.02, 0.0, 0.0, 0.5],    # Dark Blue
        [0.05, 0.0, 0.5, 1.0],    # Light Blue
        [0.75, 1.0, 1.0, 1.0],    # White
        [0.95, 1.0, 0.5, 0.2],    # Light Orange
        [1.00, 1.0, 0.3, 0.0],    # Red
    ])
    
    x = control_points[:, 0]
    r = control_points[:, 1]
    g = control_points[:, 2]
    b = control_points[:, 3]

    xq = np.linspace(0, 1, n_colors)
    rq = np.interp(xq, x, r)
    gq = np.interp(xq, x, g)
    bq = np.interp(xq, x, b)

    colors = np.vstack([rq, gq, bq]).T
    return LinearSegmentedColormap.from_list("custom_diverging", colors)

# Example usage:
cmap = custom_diverging_colormap()
plt.imshow(np.linspace(0, 1, 256)[None, :], aspect='auto', cmap=cmap)
plt.title("Custom Diverging Colormap")
plt.axis('off')
plt.show()

In [ ]:
from scipy.stats import binned_statistic_2d
import numpy as np
import matplotlib.pyplot as plt

def compute_stat(peaks_tmp, locs, bins=32):
    chan_ids, counts = np.unique(peaks_tmp['channel_index'], return_counts=True)
    locs_dict = dict(zip(range(len(locs)), locs))
    xy = np.array([locs_dict[ch] for ch in chan_ids])
    stat, xedges, yedges, _ = binned_statistic_2d(
        xy[:, 0], xy[:, 1], values=counts + 1, statistic='sum', bins=bins
    )
    return stat, xedges, yedges

#remove channel 38 from peak
channel_idx = np.where(recording_chunk.get_channel_ids() == "38")[0][0]
peaks = peaks[peaks['channel_index'] != channel_idx]
peaks2 = peaks2[peaks2['channel_index'] != channel_idx]
peaks3 = peaks3[peaks3['channel_index'] != channel_idx]
# Step 1: Compute stats
stat1, xedges, yedges = compute_stat(peaks, locs)
stat2, _, _ = compute_stat(peaks2, locs)
stat3, _, _ = compute_stat(peaks3, locs)

# Step 2: Global vmax
vmax_global = np.percentile(np.concatenate([stat1.ravel(), stat2.ravel(), stat3.ravel()]), 99)

# Step 3: Plot with same vmax
fig, axs = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for ax, stat, title in zip(axs, [stat1, stat2, stat3], 
                           ["Pre-Stim 5 min", "Stim 5 min", "Post-Stim 5 min"]):
    im = ax.imshow(stat.T, origin='lower', cmap='viridis', 
                   interpolation='bicubic', vmin= 300,vmax=vmax_global)
    ax.set_title(title)
    ax.invert_yaxis()

# Shared colorbar
cbar = fig.colorbar(im, ax=axs[2], location='right', shrink=0.7, aspect=30)
#cbar.set_label("Spike Count (+1, binned)")

# Add a legend to the right side after fig3
handles, labels = axs[2].get_legend_handles_labels()
#fig.legend(handles, labels, loc='right', bbox_to_anchor=(1.75, 0.5))
#cbar.set_label("Spike Count (+1, binned)")
plt.tight_layout()
#plt.show()
plt.savefig('./stims.png', dpi=300)



In [ ]:
# Import your custom colormap function first
def custom_diverging_colormap(n_colors=256):
    import numpy as np
    from matplotlib.colors import LinearSegmentedColormap

    control_points = np.array([
        [0.00, 0.0, 0.0, 0.0],
        [0.02, 0.0, 0.0, 0.5],
        [0.05, 0.0, 0.5, 1.0],
        [0.75, 1.0, 1.0, 1.0],
        [0.95, 1.0, 0.5, 0.2],
        [1.00, 1.0, 0.3, 0.0],
    ])
    x = control_points[:, 0]
    r = control_points[:, 1]
    g = control_points[:, 2]
    b = control_points[:, 3]
    xq = np.linspace(0, 1, n_colors)
    rq = np.interp(xq, x, r)
    gq = np.interp(xq, x, g)
    bq = np.interp(xq, x, b)
    colors = np.vstack([rq, gq, bq]).T
    return LinearSegmentedColormap.from_list("custom_diverging", colors)

# Then, plug it into your plotting code
fig, axs = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for ax, stat, title in zip(axs, [stat1, stat2, stat3], 
                           ["Pre-Stim 5 min", "Stim 5 min", "Post-Stim 5 min"]):
    im = ax.imshow(stat.T, origin='lower',
                   cmap=custom_diverging_colormap(),
                   interpolation='bicubic',
                   vmin=300, vmax=vmax_global)
    ax.set_title(title)
    ax.invert_yaxis()

# Shared colorbar
cbar = fig.colorbar(im, ax=axs[2], location='right', shrink=0.7, aspect=30)

plt.tight_layout()
plt.savefig('./stims_custom_cmap.png', dpi=300)

In [ ]:
locs_dict.keys()

In [ ]:
import mea_analysis_pipeline as msp
import helper_functions as helper

In [ ]:
#get electrode numbers
channel_ids = recording_chunk.get_channel_ids()
elec



# SpikeInterface H5 Data Extraction

In [ ]:
def full_analysis(file_path, stim_freq, trial_no, recording_electrode, stim_electrode, artifact_electrode=None):
    trial_data = StimulationAnalysis(file_path, stim_freq, recording_electrode=recording_electrode, stim_electrode=stim_electrode, artifact_electrode=artifact_electrode)

    #trial_data.plot_neuron_print()

    #trial_data.plot_individual_traces('recording', trial_no, bp_filter=True)
    #trial_data.plot_individual_traces('stim', trial_no, bp_filter=True)
    # if artifact_electrode is not None:
    #     trial_data.plot_individual_traces('artifact', trial_no, bp_filter=True, time_range=0.5)

    trial_data.get_spike_counts()
    trial_data.plot_spike_counts('recording', trial_no)
    trial_data.plot_spike_counts('stim', trial_no)
    if artifact_electrode is not None:
        trial_data.plot_spike_counts('artifact', trial_no)

    trial_data.plot_stim_traces(trial_no)

    return trial_data

# 10/23 Experiments

In [ ]:
import h5py

with h5py.File(local_path, 'r') as h5file:
    mapping = h5file['data_store/data0000/settings/mapping']
    channel_ids = [38, 820, 100]

    print("Mapping shape:", mapping.shape)
    print("Mapping dtype:", mapping.dtype)
    print("Mapping data:", mapping[:])
    # Extract the mapping data for the specified channels
    for channel_id in channel_ids:
        if channel_id < mapping.shape[0]:
            print(f"Mapping for channel {channel_id}:", mapping[channel_id])
        else:
            print(f"Channel {channel_id} not found in mapping data.")


# Trial 4 

In [ ]:
fp = '/mnt/disk20tb/PrimaryNeuronData/Maxtwo/Stimulation_MaxOnePlus_MP/250423/P002820/Trace_20250423_16_49_17.raw.h5'

trial4_data = StimulationAnalysis(fp, stim_frequency=1.0, recording_electrode=14395, stim_electrode=12627, artifact_electrode=12848)
sc = trial4_data.get_spike_counts()
trial4_data.pre_stim_length = 300
trial4_data.stim_length = 300
trial4_data.visible_artifact = True

In [ ]:
print(trial4_data.channel_dict)
print(sc.tail())

In [ ]:
trial4_data.plot_spike_counts('recording', trial_no=4)

In [ ]:
print(trial4_data.channel_dict)
start_time, end_time = trial4_data.plot_stim_traces(4, start_at=20.67, time_range=0.03)#, time_range=0.1, start_at=10.9)
peaks = trial4_data.get_spike_counts_in_range(start_time, end_time)

# Trial 3

In [ ]:
fp = '/mnt/disk15tb/nathaniel/Clean Stim N1/000185/data.raw.h5'
t3_data = StimulationAnalysis(fp, stim_frequency=0.25, recording_electrode=8165, stim_electrode=9925, artifact_electrode=9924)
t3_data.pre_stim_length = 60
t3_data.stim_length = 60
t3_sc = t3_data.get_spike_counts()


In [ ]:
print(t3_sc.tail())

In [ ]:
t3_data.plot_stim_traces(3, time_range=0.25)

In [ ]:
t3_data.visible_artifact=True
print(t3_data.channel_dict)
t3_data.plot_spike_counts('recording', trial_no=3)

In [ ]:
trial3_data = full_analysis(file_path=fp, trial_no=3, recording_electrode=8165, stim_electrode=9925, artifact_electrode=9924)


# Trial 2

In [ ]:
fp = '/mnt/disk15tb/nathaniel/Clean Stim N1/000181/data.raw.h5'

t2_data = StimulationAnalysis(fp, stim_frequency=0.25, recording_electrode=8165, stim_electrode=8173)
t2_data.pre_stim_length = 60
t2_data.stim_length = 60
t2_sc = t2_data.get_spike_counts()


In [ ]:
print(t2_sc.tail())

In [ ]:
t2_data.plot_stim_traces(2, time_range=1, start_at=12)

In [ ]:
t2_data.visible_artifact = True
t2_data.plot_spike_counts('recording', 2)

# Trial 1

In [ ]:
fp = '/mnt/disk15tb/nathaniel/Clean Stim N1/000177/data.raw.h5'

t1_data = StimulationAnalysis(fp, stim_frequency=0.25, recording_electrode=8165, stim_electrode=6405)
t1_data.pre_stim_length = 60
t1_data.stim_length = 60
t1_sc = t1_data.get_spike_counts()


In [ ]:
t1_data.plot_stim_traces(1, time_range=1)

In [ ]:
t1_data.visible_artifact = True
t1_data.plot_spike_counts('recording', 1)

In [ ]:
fp = '/mnt/disk15tb/nathaniel/000189/data.raw.h5'
recording = se.read_maxwell(fp)

channel_ids = recording.get_channel_ids()
fs = recording.get_sampling_frequency()
num_chan = recording.get_num_channels()
num_seg = recording.get_num_segments()
num_samples = recording.get_num_samples(segment_index=0)
total_recording = recording.get_total_duration()

start_time = 0
end_time = num_samples / fs

print('Sampling frequency:', fs)
print('No. of channels:', num_chan)
print('No. of segments:', num_seg)
print('total_recording:', total_recording)
print('No. of samples:', num_samples)
print(f'Start time: {start_time} seconds')
print(f'End time: {end_time} seconds')

recording_bp = si.bandpass_filter(recording, freq_min=300, freq_max=3000)

# Divide into three chunks: pre-stim, stim, post-stim

chunk_duration = total_recording / 3

samples_per_chunk = int(chunk_duration * fs)

pre_stim_start = 0
pre_stim_end = samples_per_chunk
pre_stim_data = recording_bp.get_traces(channel_ids=['572'], start_frame=pre_stim_start, end_frame=pre_stim_end)
print(recording_bp)
during_stim_start = pre_stim_end 
during_stim_end = pre_stim_end + samples_per_chunk 
during_stim_data = recording_bp.get_traces(channel_ids=['572'], start_frame=during_stim_start, end_frame=during_stim_end) 

post_stim_start = during_stim_end 
post_stim_end = num_samples 
post_stim_data = recording_bp.get_traces(channel_ids=['572'], start_frame=post_stim_start, end_frame=post_stim_end) 




In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
si.plot_probe_map(recording_bp, ax=ax, with_channel_ids=False)
ax.invert_yaxis()

# Clean Method N2

## Trial 1

In [ ]:
fp = '/mnt/disk15tb/nathaniel/Clean Stim N2/000228/data.raw.h5'
n2_t1_data = StimulationAnalysis(fp, recording_electrode=13306, stim_electrode=14846)
print(f'Stim Channel: {n2_t1_data.stim_channel}')
print(f'Recording Channel: {n2_t1_data.rec_channel}')
#print(n2_t1_data.get_spike_counts())
n2_t1_data.plot_stim_traces(1, bp_filter=True, time_range=2, start_at=20)
full_analysis(fp, 1, recording_electrode=13306, stim_electrode=14846)



## Trial 2

In [ ]:
fp = '/mnt/disk15tb/nathaniel/Clean Stim N2/000226/data.raw.h5'
n2_t1_data = StimulationAnalysis(fp, recording_electrode=13306, stim_electrode=13313)
print(f'Stim Channel: {n2_t1_data.stim_channel}')
print(f'Recording Channel: {n2_t1_data.rec_channel}')
#print(n2_t1_data.get_spike_counts())
n2_t1_data.plot_stim_traces(2, bp_filter=True, time_range=3, start_at=1)
full_analysis(fp, 1, recording_electrode=13306, stim_electrode=13313)

## Trial 3

In [ ]:
fp = '/mnt/disk15tb/nathaniel/Clean Stim N2/000230/data.raw.h5'
n2_t1_data = StimulationAnalysis(fp, recording_electrode=13306, stim_electrode=13299)
print(f'Stim Channel: {n2_t1_data.stim_channel}')
print(f'Recording Channel: {n2_t1_data.rec_channel}')
#print(n2_t1_data.get_spike_counts())
n2_t1_data.plot_stim_traces(3, bp_filter=True, time_range=3, start_at=1)
full_analysis(fp, 3, recording_electrode=13306, stim_electrode=13299)

## Trial 4

In [ ]:
fp = '/mnt/disk15tb/nathaniel/Clean Stim N2/000233/data.raw.h5'
n2_t1_data = StimulationAnalysis(fp, recording_electrode=13306, stim_electrode=11766)
print(f'Stim Channel: {n2_t1_data.stim_channel}')
print(f'Recording Channel: {n2_t1_data.rec_channel}')
#print(n2_t1_data.get_spike_counts())
n2_t1_data.plot_stim_traces(4, bp_filter=True, time_range=0.1, start_at=18.38)
n2_t1_data.plot_stim_traces(4, bp_filter=True, time_range=0.1, start_at=18.2)
full_analysis(fp, 4, recording_electrode=13306, stim_electrode=11766)

# Dirty Method N2

In [ ]:
fp = '/mnt/disk15tb/nathaniel/Unclean Stim N2/000236/data.raw.h5'
n2_t1_data = StimulationAnalysis(fp, recording_electrode=13077, stim_electrode=13525)
print(f'Stim Channel: {n2_t1_data.stim_channel}')
print(f'Recording Channel: {n2_t1_data.rec_channel}')
print(n2_t1_data.get_spike_counts())
n2_t1_data.plot_stim_traces(1, bp_filter=True, time_range=0.075, start_at=0.15)
full_analysis(fp, 1, recording_electrode=13077, stim_electrode=13525)